# VCuboidFIT trên Kaggle: Tầng 0 + Tầng 1
Chạy lần lượt từ trên xuống. Dùng Google Colab thì mở `run_on_colab.ipynb`.

**Cài đặt notebook trước khi chạy** (Settings, cột bên phải):
- *Accelerator*: **GPU T4 x2** (không dùng P100: sm_60 không được hỗ trợ). Tầng 0 chạy được không cần GPU.
- *Internet*: **On** (cần để `git clone`, `pip`, tải OpenPCDet). Kaggle yêu cầu xác minh số điện thoại mới bật được.
- *Add Input*: gắn một Kaggle Dataset riêng tư chứa nuScenes (`v1.0-mini/`, `samples/`, `sweeps/`), **hoặc** dán link tải vào `SOURCE_URL` ở phần 1.

| Phần | Việc |
|---|---|
| 1 | Tìm dữ liệu nuScenes (Dataset đã gắn, hoặc tải từ `SOURCE_URL`) |
| 2-4 | Cấu hình, kiểm GPU, cài môi trường |
| 5-7 | Tầng 0 → Tầng 1 → Tầng 0 lần hai + báo cáo |
| 8 | Lưu kết quả vào `/kaggle/working` (tải ở tab Output) |

Kaggle không giữ gì giữa các phiên (trừ khi *Save Version*), nên mỗi lần chạy cài lại môi trường (~10-15 phút). Hướng dẫn: `docs/kaggle-colab.md`.

## 1. Dữ liệu nuScenes
Thứ tự: Dataset đã gắn trong `/kaggle/input` → `SOURCE_URL` (nuscenes.org: chuột phải nút tải Mini → *Copy link address*, link chỉ sống ít phút) → file `.tgz`/`.zip` nằm trong Dataset gắn vào.

In [ ]:
# Ô 1 - NẠP DỮ LIỆU (Kaggle)
import glob, os, shutil, subprocess, tarfile, zipfile

SOURCE_URL = ""                 # tuỳ chọn: link tải trực tiếp
DEST = "/tmp/nuscenes"          # ổ tạm (rộng hơn /kaggle/working, không bị tính vào Output)


def _safe(root, name):
    out = os.path.realpath(os.path.join(root, name))
    assert out == root or out.startswith(root + os.sep), f"đường dẫn không an toàn: {name}"
    return out


def _find_nusc(base, depth=3):
    """Thư mục (base hoặc con trong `depth` cấp) chứa v1.0-*, hoặc None."""
    for d in range(depth + 1):
        hit = [h for h in glob.glob(os.path.join(base, *["*"] * d, "v1.0-*")) if os.path.isdir(h)]
        if hit:
            return os.path.dirname(sorted(hit)[0])
    return None


def _extract(path, root):
    n = 0
    if tarfile.is_tarfile(path):
        with tarfile.open(path) as tf:
            for m in tf:
                if not (m.isfile() or m.isdir()):
                    continue
                _safe(root, m.name)
                tf.extract(m, root)
                n += 1
    elif zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as zf:
            for m in zf.infolist():
                name = m.filename[5:] if m.filename.startswith("data/") else m.filename  # vcf-pack
                if m.is_dir() or not name:
                    continue
                out = _safe(root, name)
                os.makedirs(os.path.dirname(out), exist_ok=True)
                with zf.open(m) as s, open(out, "wb") as d:
                    shutil.copyfileobj(s, d)
                n += 1
    else:
        print("bỏ qua (không phải tar/zip):", path)
    return n


def _download(url, target):
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if "drive.google.com" in url:
        import gdown
        gdown.download(url, target, quiet=False, fuzzy=True)
    else:
        r = subprocess.run(["wget", "-q", "--show-progress", "-O", target, url])
        assert r.returncode == 0, "Tải thất bại: link nuscenes.org chỉ sống ít phút, hãy copy link mới"
    return target


def _unpack(files, dest):
    os.makedirs(dest, exist_ok=True)
    root = os.path.realpath(dest)
    total = sum(_extract(f, root) for f in files)
    found = _find_nusc(dest)
    assert found, f"Giải nén {total} file nhưng không thấy thư mục v1.0-* trong {dest}"
    print(f"Đã giải nén {total} file từ {len(files)} tệp")
    return found


NUSC_READY = _find_nusc("/kaggle/input") or _find_nusc(DEST)
if NUSC_READY:
    print("Đã có dữ liệu ở", NUSC_READY)
elif SOURCE_URL.strip():
    NUSC_READY = _unpack([_download(SOURCE_URL.strip(), "/tmp/download/data.bin")], DEST)
else:
    archives = sorted(glob.glob("/kaggle/input/**/*.tgz", recursive=True)
                      + glob.glob("/kaggle/input/**/*.tar", recursive=True)
                      + glob.glob("/kaggle/input/**/*.zip", recursive=True))
    assert archives, ("Không thấy nuScenes: gắn Dataset chứa v1.0-mini/ (Add Input) hoặc dán "
                      "link vào SOURCE_URL")
    NUSC_READY = _unpack(archives, DEST)
print("NUSC_READY =", NUSC_READY, "|", sorted(os.listdir(NUSC_READY)))


## 2. Cấu hình
`EXP` ở `/kaggle/working` (tải về từ tab Output hoặc *Save Version*). `GIT_REF` đổi thành `main` sau khi nhánh được merge.

In [ ]:
# Ô 2 - cấu hình (Kaggle)
import os, sys, glob, shutil, subprocess
from pathlib import Path

GIT_URL = "https://github.com/Nguyendev037/VCuboidFit.git"
GIT_REF = "fix/Linh"
NUSC = NUSC_READY
WORK = Path("/kaggle/working")
EXP = str(WORK / "exp")
Path(EXP).mkdir(parents=True, exist_ok=True)

REPO = WORK / "vcf"
if not (REPO / "model" / "scripts" / "tier1.sh").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", GIT_REF, GIT_URL, str(REPO)], check=True)
REPO = str(REPO / "model")
os.environ.update(NUSC=NUSC, EXP=EXP, REPO=REPO)
print("NUSC =", NUSC, "\nEXP  =", EXP, "\nREPO =", REPO)


## 3. Kiểm GPU
Cần CUDA 12.x có sẵn trong image Kaggle/Colab. **P100 (sm_60) không nằm trong danh sách arch** ⇒ đổi sang T4 (Kaggle: Accelerator → *GPU T4 x2*).

In [ ]:
import torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "không có nvidia-smi")
print("torch", torch.__version__, "cuda", torch.version.cuda, "available", torch.cuda.is_available())
assert torch.cuda.is_available(), "Không có GPU: bật Accelerator GPU rồi chạy lại (Tầng 0 chạy CPU được: bỏ qua phần 3, 4 và 6)."
CAP = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "sm_%d%d" % CAP)
if CAP < (7, 0):
    raise SystemExit("GPU sm_%d%d (vd P100) KHÔNG được hỗ trợ — chọn T4/L4/A100." % CAP)
ARCH = "%d.%d" % CAP          # build CUDA ops đúng GPU đang chạy (T4=7.5, A100=8.0, L4=8.9)
os.environ["TORCH_CUDA_ARCH_LIST"] = ARCH
print("TORCH_CUDA_ARCH_LIST =", ARCH)

## 4. Cài môi trường
Dùng `scripts/colab_setup.sh` (cũng chạy trên Kaggle): cài thư viện và build OpenPCDet vào `/tmp/vcf_home`. Nếu báo cần khởi động lại vì `numpy<2`, dùng *Run → Restart & Run All*.

In [ ]:
# Ô 4 - cài môi trường qua scripts/colab_setup.sh
VCF_HOME = "/tmp/vcf_home"
os.makedirs(VCF_HOME, exist_ok=True)
os.environ["VCF_HOME"] = VCF_HOME
SETUP_SH = f"{REPO}/scripts/colab_setup.sh"
assert os.path.isfile(SETUP_SH), f"Không thấy {SETUP_SH}: clone sai nhánh? (GIT_REF)"
# Colab không hiện stdout của tiến trình con nếu để mặc định, nên đọc từng dòng rồi in ra ô này.
proc = subprocess.Popen(["bash", SETUP_SH], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
tail = []
for line in proc.stdout:
    print(line, end="")
    tail = (tail + [line])[-25:]
rc = proc.wait()
assert rc == 0, f"colab_setup.sh lỗi (mã {rc}). 25 dòng cuối:" + chr(10) + "".join(tail)
PCDET_ROOT = f"{VCF_HOME}/OpenPCDet"
os.environ["PCDET_ROOT"] = PCDET_ROOT
os.environ["PYTHONPATH"] = f"{REPO}/worker:{PCDET_ROOT}:" + os.environ.get("PYTHONPATH", "")
for p in (f"{REPO}/worker", PCDET_ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)
import importlib; importlib.invalidate_caches()
import pcdet, pcdet.datasets
print("pcdet", pcdet.__version__, "| PCDET_ROOT =", PCDET_ROOT)


## 5. Tầng 0 (CPU): descriptor hình học, chọn 5%, tune
Tạo `EXP/index.parquet` + `features/` — **bắt buộc** trước Tầng 1 (`tier1.sh` thoát mã 4 nếu thiếu).

In [ ]:
%%bash
set -e
cd "$REPO/worker"
python -m c4.cli.lidar_experiment --data-root "$NUSC" --out "$EXP" --split V --tune

## 6. Tầng 1 (GPU): PointPillars huấn luyện từ seed S + suy luận
`--sweeps 1` (chỉ keyframe — ghi rõ trong báo cáo, dùng cùng giá trị cho downstream). Hết VRAM: `--batch 2` rồi `1`. Checkpoint ở `EXP/t1/ckpt/seed_latest.pth`; chạy lại thì phần đã có được bỏ qua.

In [ ]:
%%bash
set -e
bash "$REPO/scripts/tier1.sh" "$NUSC" "$EXP" --no-docker --sweeps 1 --epochs 20 --batch 4

## 7. Chạy lại Tầng 0 để có ma trận Tầng 1
Lần này có `t1/signals.parquet` ⇒ ma trận thêm `hybrid_mmr`, `t1_*_mmr`, `entropy_only`.

In [ ]:
%%bash
set -e
cd "$REPO/worker"
python -m c4.cli.lidar_experiment --data-root "$NUSC" --out "$EXP" --split V --tune

In [ ]:
from IPython.display import Markdown, display
rep = Path(EXP) / "V" / "report_selection.md"
display(Markdown(rep.read_text(encoding="utf-8")) if rep.exists() else Markdown(f"Chưa có `{rep}`"))

## 8. Lưu kết quả
Zip `EXP` **không kèm** `t1/pcdet_data`, đặt trong `/kaggle/working` ⇒ tải ở tab **Output** hoặc *Save Version → Save & Run All (Commit)*.

**Resume:** Kaggle không giữ gì giữa các phiên: mở lại, chạy phần 1 → 4, giải nén zip cũ vào `EXP` nếu cần, rồi chạy tiếp phần 5–7.

In [ ]:
import zipfile, time
OUT_DIR = WORK
zpath = OUT_DIR / f"vcf_exp_{time.strftime('%Y%m%d_%H%M')}.zip"
skip = Path(EXP) / "t1" / "pcdet_data"
n = 0
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(EXP, followlinks=False):
        dirs[:] = [d for d in dirs if Path(root, d) != skip]
        for f in files:
            fp = Path(root, f)
            if fp.is_symlink():
                continue
            z.write(fp, fp.relative_to(Path(EXP).parent)); n += 1
print(f"{zpath}  {zpath.stat().st_size/1e6:.1f} MB, {n} file")

## 10. (Tuỳ chọn, mặc định tắt) Cổng G1
Đặt `RUN_G1 = True` để chạy `c4.cli.lidar_g1`. Chỉ bật khi đã đọc SPEC của cổng G1.

In [ ]:
RUN_G1 = False
if RUN_G1:
    r = subprocess.run([sys.executable, "-m", "c4.cli.lidar_g1", "--data-root", NUSC, "--out", EXP],
                       cwd=f"{REPO}/worker")
    print("exit", r.returncode)
else:
    print("G1 tắt (RUN_G1 = False)")